# 101 · Data science perspective lab

Companion to [Data science perspective](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/101/data-science-perspective/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/101/data_science_perspective.ipynb)

**Goal:** feel the difference between CSV/JSONL, pickle trust boundaries, and row vs columnar layout for analytics-style access.

**Why this lab:** data work often “just works” with pickle or CSV until scale, multi-language handoff, or a security review. You need a tactile map of trade-offs before choosing a lake or feature-store format.

**How to use:** run cells top to bottom (stdlib only). Optional `pyarrow` cell upgrades the same ideas to Parquet/Arrow if installed.

**Expect:** CSV loses types; JSONL keeps types but is large; pickle is compact but trust-sensitive; column layout makes single-column scans cheaper than row dicts in a toy model.

> **Honesty banner:** sizes and timings here are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth.


In [ ]:
import csv
import io
import json
import pickle
import struct
from pathlib import Path

# Tiny "lake" sample — wide enough to show column projection idea
ROWS = [
    {"event_id": i, "user": f"u{i%5}", "feature_a": i * 0.1, "feature_b": i % 3,
     "label": "pos" if i % 4 == 0 else "neg", "payload_note": "x" * 8}
    for i in range(100)
]
print(len(ROWS), "rows; columns:", list(ROWS[0]))



## CSV as a lossy social format

**Why:** CSV is the universal export people reach for—yet it is a *social* format, not a typed contract.

**How:** write a small table to CSV and read it back with `csv.DictReader`.

**Expect:** every value comes back as a **string** (e.g. `feature_a` was `float`, becomes `str`). Nulls, dates, and nested data stay ambiguous.

**Why it matters:** fine at the edge (spreadsheets, quick dumps); a poor system of record for lakes and ML pipelines where types and nulls must survive years.


In [ ]:
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=list(ROWS[0]))
w.writeheader()
w.writerows(ROWS)
csv_text = buf.getvalue()
print("CSV nbytes:", len(csv_text.encode()))
print(csv_text.splitlines()[0])
print(csv_text.splitlines()[1])
# Types are gone — everything is text on reload
buf.seek(0)
reread = list(csv.DictReader(buf))
print("feature_a type after CSV round-trip:", type(reread[0]["feature_a"]), reread[0]["feature_a"])



## JSONL — append-friendly landing zone

**Why:** one JSON object per line is a pragmatic compromise between “human-ish text” and batch/stream processing.

**How:** serialize the same rows as JSON Lines and parse the first line.

**Expect:** types survive (`float` stays numeric); file is larger than CSV because keys repeat every line.

**Why it matters:** good landing zone / log / intermediate format—not a substitute for columnar storage when you scan few columns over huge tables.


In [ ]:
jsonl = "\n".join(json.dumps(r, separators=(",", ":")) for r in ROWS).encode()
print("JSONL nbytes:", len(jsonl))
first = json.loads(jsonl.splitlines()[0])
print("feature_a type after JSONL:", type(first["feature_a"]), first["feature_a"])



## Pickle trust boundary

**Why:** pickle is maximum Python convenience and a classic **trust** mistake when bytes leave your process.

**How:** round-trip trusted data with `pickle.dumps` / `loads`. We only show the byte prefix—not a live exploit.

**Expect:** compact bytes and perfect Python fidelity for the trusted path. The rule is policy, not size.

**Rule:** if bytes are not from a fully trusted source you control → **do not unpickle**.

**Why it matters:** untrusted pickle can become code execution. Portable formats (JSONL, Parquet, Arrow, schema binary) are interchange; pickle is an in-trust-domain tool.


In [ ]:
trusted = pickle.dumps(ROWS)
print("pickle nbytes:", len(trusted))
assert pickle.loads(trusted)[0]["event_id"] == 0

# Illustrative hostile pattern (do NOT run on untrusted bytes in production).
# pickle can invoke callables during load — we only show the *opcode surface*, not a live exploit.
print("pickle protocol opcodes (prefix):", trusted[:20])
print("OK: portable formats (JSONL/Parquet/Arrow) for multi-language or untrusted interchange")



## Row vs columnar access cost (toy model)

**Why:** analytics rarely needs whole rows—often one or few columns over many records.

**How:** same data as list-of-dicts (row) vs dict-of-lists (column); time summing `feature_a`.

**Expect:** the column path is faster here because it walks a dense list, not pointer-rich dicts. Real engines also skip unread columns on disk (Parquet/Arrow idea).

**Why it matters:** “we already have events as JSON/Protobuf” is a weak lake design if the workload is wide-table scans.


In [ ]:
# Row store: list of dicts (pointer-rich)
row_store = ROWS

# Column store: one list per column
col_store = {k: [r[k] for r in ROWS] for k in ROWS[0]}


def sum_feature_a_rows():
    return sum(r["feature_a"] for r in row_store)


def sum_feature_a_cols():
    return sum(col_store["feature_a"])


import timeit
from statistics import median

t_row = median(timeit.repeat(sum_feature_a_rows, number=2000, repeat=5))
t_col = median(timeit.repeat(sum_feature_a_cols, number=2000, repeat=5))
assert abs(sum_feature_a_rows() - sum_feature_a_cols()) < 1e-9
print(f"sum feature_a via rows: {t_row*1e3:.3f} ms median")
print(f"sum feature_a via cols: {t_col*1e3:.3f} ms median")
print("Columnar wins more as width grows and engines skip unread columns (Parquet/Arrow idea).")



## Optional: Parquet/Arrow if pyarrow is installed

**Why:** connect the toy column model to formats you will actually see in production data planes.

**How:** if `pyarrow` is available, write Parquet and read only `feature_a`.

**Expect:** SKIP message without pyarrow; with it, a compact file and a single-column read.

**Why it matters:** common pattern—**Parquet on disk / Arrow in memory** between engines—not “one RPC codec for the lake.”


In [ ]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq

    table = pa.Table.from_pylist(ROWS)
    sink = pa.BufferOutputStream()
    pq.write_table(table, sink, compression="zstd")
    parquet_bytes = sink.getvalue().to_pybytes()
    print("Parquet nbytes:", len(parquet_bytes))
    # Project one column only
    col_only = pq.read_table(pa.BufferReader(parquet_bytes), columns=["feature_a"])
    print("read feature_a only:", col_only.column(0).to_pylist()[:5], "…")
except ImportError:
    print("SKIP pyarrow — optional: pip install pyarrow")
    print("Pattern to remember: Parquet on disk / Arrow in memory between engines.")



## Takeaways

| Workload | Prefer | Why |
|----------|--------|-----|
| Human export | CSV (edge only) | Social interoperability, not types |
| Landing / logs | JSONL | Typed-ish, append-friendly |
| Analytics scans | Columnar (Parquet-class) | Read few columns, compress well |
| Engine handoff | Arrow | Shared in-memory layout |
| Untrusted / multi-lang | Never pickle | Trust + portability |

**Why it matters overall:** format choice is a workload and trust decision, not a convenience default from the first notebook cell.

**Next:** [Engineering mini lab](./engineering_perspective.ipynb) · [Serialization 201](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/)
